# Lesson 02 - Microsoft Agent Frameworkin tutkiminen

**Microsoft Agent Framework (MAF)** on yhtenäinen kehys tekoälyagenttien rakentamiseen. Se tarjoaa siistin, koostettavan arkkitehtuurin, jossa on neljä keskeistä rakennuspalikkaa:

- **Client** – yhdistää tekoälymallin päätepisteeseen ja hoitaa viestinnän
- **Agent** – käärii clientin ohjeilla ja työkalumääritelmillä
- **Tools** – laajentavat agentin kykyjä mallin kutsuttavilla mukautetuilla toiminnoilla
- **Session** – ylläpitää keskusteluhistoriaa monivaiheisissa vuorovaikutuksissa

Tässä oppitunnissa rakennamme **matkavarausalustan agentin**, joka tarkistaa kohteen saatavuuden näiden käsitteiden avulla.


## Asennus


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Ymmärtäminen Agenttikehyksen Arkkitehtuurista

Microsoft Agent Framework noudattaa kerroksellista arkkitehtuuria:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Asiakas** – `AzureAIProjectAgentProvider` muodostaa yhteyden Azure OpenAI -käyttöönottoon. Se hoitaa todennuksen, pyyntöjen muotoilun ja vastausten jäsentämisen.
2. **Agentti** – Luodaan asiakkaasta `provider.create_agent()`-menetelmällä, agentti yhdistää mallin käytön ohjeistuksiin (järjestelmän kehotus) ja työkaluihin.
3. **Työkalut** – Python-funktioita, jotka on koristeltu `@tool`-koristeella ja joita agentti voi kutsua toimenpiteiden suorittamiseksi tai datan hakemiseksi.
4. **Istunto** – `AgentSession`-objekti (luodaan `agent.create_session()`-menetelmällä), joka tallentaa keskusteluhistorian mahdollistaen monivuorokeskustelun, jossa agentti muistaa aiemman kontekstin.

Rakennetaan jokainen kerros askel askeleelta.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Työkalujen lisääminen @tool-dekoraattorilla

Työkalut antavat agenteille mahdollisuuden suorittaa toimintoja tekstin generoinnin lisäksi. `@tool`-dekoraattori muuttaa tavallisen Python-funktion sellaiseksi, jota agentti voi kutsua.

Tärkeimmät kohdat:
- Käytä `Annotated[type, "description"]`, jotta malli ymmärtää jokaisen parametrin.
- Dokumentaatioteksti (docstring) toimii työkalun kuvauksena, jonka malli näkee.
- `approval_mode="never_require"` tarkoittaa, että työkalu suoritetaan automaattisesti ilman käyttäjän vahvistusta.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Agentin luominen työkaluilla

Nyt yhdistämme asiakkaan, ohjeet ja työkalut agentiksi. `instructions` toimii järjestelmän kehotteena — ne määrittelevät agentin persoonallisuuden ja käyttäytymisen.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Monivuorokeskustelut istuntojen kanssa

`AgentSession` (luotu `agent.create_session()` -kutsulla) seuraa kaikkia viestejä keskustelussa. Antamalla saman istunnon jokaiseen `agent.run()` -kutsuun agentti saa käyttöönsä koko keskusteluhistorian ja voi viitata aiempiin viesteihin.

Annamme `tools=[check_destination_availability]`, jotta agentti voi kutsua saatavuustarkistustamme jokaisen vuoron aikana.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Yhteenveto

Tässä oppitunnissa tutustuit Microsoft Agent Frameworkin neljään peruspilariin:

| Käsite | Mitä opit |
|---------|------------------|
| **Asiakas** | `AzureAIProjectAgentProvider` yhdistää Azure OpenAI:hin tunnistetietopohjaisella todennuksella |
| **Agentti** | `provider.create_agent()` yhdistää malliyhteyden ohjeistuksiin ja nimeen |
| **Työkalut** | `@tool`-koristaja altistaa Python-funktiot agentin kutsuttaviksi |
| **Istunto** | `agent.create_session()` säilyttää keskusteluhistorian useiden vuorojen ajan |

Nämä rakennuspalikat muodostavat yhdessä agenteja, jotka pystyvät käymään luonnollisia keskusteluja, kutsumaan ulkoisia funktioita ja ylläpitämään kontekstia — perusta kehittyneemmille agenttipatterneille myöhemmissä oppitunneissa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Pyrimme tarkkuuteen, mutta ota huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja omalla kielellään on auktorisoitu lähde. Tärkeiden tietojen osalta suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
